[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helvecioneto/datascience/blob/main/praticas/pratica03.ipynb)

# Aula Prática 03 — Mineração de Textos: o cliente recomendaria o produto?

**Disciplina:** Introdução à Ciência de Dados (NSA A110) · **Professor:** Helvecio Bezerra Leal Neto · UFOPA
**Data:** 29/10/2026, 19h às 22h · **Modalidade:** prática em equipes (2 a 3 pessoas)

### Como usar este notebook
Este notebook é um **roteiro**. Leia cada bloco de texto **antes** de rodar a célula de código logo abaixo dele. Os ícones dizem o que fazer:

| Ícone | O que você faz |
|---|---|
| ▶️ **Rode** | A célula está pronta. Execute com `Shift + Enter` e leia a saída. |
| 🧭 **Complete** | A célula tem lacunas `___`. Troque cada uma pelo que o texto pede e rode. Se travar, abra o **💡 Ver resposta** logo abaixo, mas tente antes. |
| ✍️ **Responda** | Escreva a resposta da equipe na célula de texto indicada (dois cliques para editar). |

Rode as células **na ordem**, de cima para baixo. Se algo der errado, use *Ambiente de execução → Reiniciar e executar tudo*.

### Objetivos de aprendizagem
Ao final da prática, a equipe será capaz de:
- **Transformar** textos em uma tabela de números com o **saco de palavras** (*bag-of-words*).
- **Separar** treino e teste **antes** de ajustar o vetorizador, evitando o vazamento de dados.
- **Treinar** e **ler** uma **árvore de decisão** que prevê se o cliente recomenda o produto.
- **Avaliar** o modelo com matriz de confusão, precisão e revocação, comparando com uma linha de base, e **reconhecer** o sobreajuste ao variar a profundidade da árvore.
- **Agrupar** avaliações sem rótulo com **k-means** e dar nome aos grupos encontrados.

### Pré-requisitos
- Aula 07: aprendizado supervisionado e não supervisionado, treino e teste, vazamento de dados, sobreajuste, matriz de confusão, precisão, revocação e árvore de decisão.
- Prática 01 e 02: pandas básico (`DataFrame`, `head`, `value_counts`) e o ciclo do CRISP-DM.

## O problema

Vocês foram contratados pela equipe de atendimento (**SAC**) de uma grande loja online. Todo dia chegam milhares de avaliações de clientes e ninguém consegue ler todas. As que mais importam são as dos clientes **insatisfeitos**: com elas o SAC age rápido, troca o produto, devolve o dinheiro ou cobra o fornecedor.

O SAC tem **duas perguntas**:

> **Pergunta 1:** lendo só o **texto** da avaliação, conseguimos prever se o cliente **recomendaria** o produto a um amigo?
>
> **Pergunta 2:** entre os clientes insatisfeitos, **do que** eles reclamam?

**Os dados.** Vamos usar a base pública **B2W-Reviews01**, com cerca de 130 mil avaliações **reais** de clientes da Americanas.com, publicada pela B2W Digital ([repositório](https://github.com/americanas-tech/b2w-reviews01), licença CC BY-NC-SA 4.0). Além do texto, cada cliente respondeu à pergunta *"Você recomendaria este produto a um amigo?"*. Essa resposta é o nosso **rótulo**: ninguém precisou rotular nada à mão.

**Que tipo de problema é cada pergunta?** Na linguagem da Aula 07:

| Pergunta | Tipo de aprendizado | Técnica de hoje |
|---|---|---|
| 1. Recomenda ou não? | **supervisionado** (temos o rótulo), de **classificação** | árvore de decisão |
| 2. Do que reclamam? | **não supervisionado** (não há rótulo de "assunto") | clusterização com k-means |

Na Pergunta 1, o **X** (entrada) é o texto escrito pelo cliente e o **y** (rótulo) é a resposta `sim` ou `não`.

### O caminho que vamos seguir
1. Carregar e conhecer os dados
2. Preparar os dados
3. Entender como o computador "lê" texto: o saco de palavras
4. Separar treino e teste
5. Vetorizar (texto → números)
6. Treinar uma árvore de decisão e ler as regras dela
7. Avaliar o modelo
8. Escolher a profundidade da árvore (sobreajuste)
9. Olhar os erros do modelo
10. Agrupar as reclamações com k-means
11. Registrar tudo no CRISP-DM

## Etapa 0 · Preparar o ambiente

▶️ **Rode** as duas células abaixo. A primeira instala as bibliotecas (no Colab elas já vêm instaladas e a célula termina rápido). A segunda importa tudo o que vamos usar.

In [ ]:
# pandas: tabelas · matplotlib: gráficos · scikit-learn: vetorizadores, modelos e métricas
%pip install -q pandas matplotlib scikit-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

SEMENTE = 42                               # mesma semente = mesmos resultados para todas as equipes
pd.set_option("display.max_colwidth", 200) # mostra textos mais longos nas tabelas
print("Tudo pronto!")

## Etapa 1 · Carregar e conhecer os dados
*CRISP-DM, fase 2: entendimento dos dados.*

▶️ **Rode** a célula abaixo. Ela baixa o arquivo CSV direto do GitHub (cerca de 50 MB, leva alguns segundos), guarda só as colunas que interessam e traduz os nomes para o português.

In [ ]:
URL = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"

colunas = {
    "review_title": "titulo",
    "review_text": "texto",
    "overall_rating": "nota",
    "recommend_to_a_friend": "recomenda",
    "site_category_lv1": "categoria",
}
df = pd.read_csv(URL, usecols=list(colunas), low_memory=False).rename(columns=colunas)
df = df[["titulo", "texto", "nota", "recomenda", "categoria"]]   # ordem mais fácil de ler
print("Avaliações carregadas:", len(df))

🧭 **Complete:** um `DataFrame` tem um método que mostra as **primeiras linhas** e um atributo que diz **quantas linhas e colunas** ele tem. Vocês usaram os dois na Prática 01. Troque os `___` e rode.

In [ ]:
print("Linhas e colunas:", df.___)
df.___()

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
print("Linhas e colunas:", df.shape)
df.head()
```
</details>

✍️ **Responda** na célula abaixo, olhando a tabela:
1. Qual coluna será o **X** e qual será o **y** da Pergunta 1?
2. A coluna `nota` (1 a 5 estrelas) parece muito ligada a `recomenda`. Por que **não** devemos usá-la como entrada do modelo? *(Dica: se o SAC já tivesse a nota, ainda precisaria de um modelo? Pensem também no vazamento de dados da Aula 07.)*

**✍️ Respostas da equipe**

1.
2.

## Etapa 2 · Preparar os dados
*CRISP-DM, fase 3: preparação dos dados.*

Antes de modelar, três cuidados:
1. **Tirar linhas incompletas:** avaliação sem texto ou sem resposta de recomendação não serve para treinar.
2. **Traduzir o rótulo:** `Yes`/`No` viram `sim`/`não`.
3. **Usar uma amostra:** 130 mil textos deixariam a aula lenta. Sorteamos **20 mil** avaliações, o bastante para aprender bem. A semente garante que todas as equipes sorteiem as mesmas.

▶️ **Rode.**

In [ ]:
df = df.dropna(subset=["texto", "recomenda"])
df["recomenda"] = df["recomenda"].map({"Yes": "sim", "No": "não"})
df = df.sample(n=20_000, random_state=SEMENTE).reset_index(drop=True)

print("Avaliações na amostra:", len(df))
df[["texto", "recomenda"]].head(3)

### As classes estão equilibradas?
Na Aula 07 vimos que a **acurácia engana** quando uma classe é muito mais comum que a outra. Precisamos saber a proporção de `sim` e `não`.

🧭 **Complete:** o método do pandas que **conta quantas vezes cada valor aparece** numa coluna. Com `normalize=True` ele devolve proporções no lugar de contagens.

In [ ]:
print(df["recomenda"].___())
print()
print(df["recomenda"].___(normalize=True).round(3))

df["recomenda"].value_counts().plot(kind="bar", color=["tab:green", "tab:red"], rot=0,
                                    title="Recomendaria o produto?")
plt.ylabel("avaliações")
plt.show()

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
print(df["recomenda"].value_counts())
print(df["recomenda"].value_counts(normalize=True).round(3))
```
</details>

✍️ **Responda:** imagine um "modelo preguiçoso" que **não lê nada** e responde sempre `sim`. Qual seria a acurácia dele? Esse número é a nossa **linha de base**: qualquer modelo de verdade precisa ser **melhor** que ele.

**✍️ Resposta da equipe:** a linha de base é de ____ %.

▶️ **Rode** para ler algumas avaliações de clientes que **não** recomendariam o produto. Ler os dados é parte do trabalho: repare no vocabulário, nos erros de digitação e nas gírias.

In [ ]:
df.loc[df["recomenda"] == "não", ["titulo", "texto"]].sample(5, random_state=1)

## Etapa 3 · Como o computador "lê" texto? O saco de palavras

Os modelos de aprendizado de máquina só aceitam **números**. Então precisamos transformar cada texto numa linha de uma tabela. A ideia mais simples é o **saco de palavras** (*bag-of-words*):

- cada **coluna** é uma palavra do vocabulário;
- cada **linha** é um texto;
- cada **célula** conta quantas vezes aquela palavra aparece naquele texto.

O `CountVectorizer` do scikit-learn faz isso por nós. No caminho, ele **tokeniza** (quebra o texto em palavras) e **normaliza** a caixa (tudo em minúsculas).

▶️ **Rode** o exemplo com três frases curtas.

In [ ]:
frases = [
    "Adorei o produto, chegou rápido",
    "Não gostei, o produto chegou quebrado",
    "Produto bom, recomendo. Recomendo muito!",
]

cv = CountVectorizer()
matriz = cv.fit_transform(frases)          # aprende o vocabulário E transforma as frases

pd.DataFrame(matriz.toarray(), columns=cv.get_feature_names_out(), index=frases)

✍️ **Responda** olhando a tabela:
1. Quantas colunas (palavras) o vocabulário tem? Por que `Produto` e `produto` viraram uma coluna só?
2. A palavra **"o"** sumiu. Por quê? *(Dica: o `CountVectorizer`, por padrão, ignora "palavras" com uma letra só.)*
3. Por que a célula `recomendo` da terceira frase vale 2?

**✍️ Respostas da equipe**

1.
2.
3.

### O ponto fraco do saco de palavras
O nome não é por acaso: as palavras são jogadas num **saco** e a **ordem se perde**.

▶️ **Rode** e compare duas frases com sentidos opostos.

In [ ]:
duas = ["gostei, não recomendo", "não gostei, recomendo"]
cv2 = CountVectorizer()
print(pd.DataFrame(cv2.fit_transform(duas).toarray(), columns=cv2.get_feature_names_out(), index=duas))

✍️ **Responda:** as duas linhas ficaram iguais. O que isso significa para um modelo que só vê essa tabela? Guardem essa ideia: vamos voltar a ela na Etapa 9.

**✍️ Resposta da equipe:**

## Etapa 4 · Separar treino e teste (antes de tudo!)
*CRISP-DM, fases 4 e 5: modelagem e avaliação.*

Na Aula 07 vimos a regra de ouro: o modelo é avaliado em dados que ele **nunca viu**, como uma prova com questões novas. Por isso separamos agora, **antes** de qualquer ajuste:

- **treino (75%)**: o modelo aprende com estes textos;
- **teste (25%)**: ficam guardados e só servem para a nota final.

O parâmetro `stratify` mantém a mesma proporção de `sim` e `não` nos dois pedaços.

🧭 **Complete:** `test_size` recebe a **fração** que vai para o teste (um número entre 0 e 1), e `stratify` recebe a variável com os **rótulos**.

In [ ]:
X = df["texto"]        # entrada: os textos
y = df["recomenda"]    # rótulo: sim / não

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=___, stratify=___, random_state=SEMENTE
)

print("Treino:", len(X_treino), "avaliações")
print("Teste: ", len(X_teste), "avaliações")
print("Proporção de 'não' no treino:", round((y_treino == "não").mean(), 3))
print("Proporção de 'não' no teste: ", round((y_teste == "não").mean(), 3))

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEMENTE
)
```
</details>

## Etapa 5 · Vetorizar: `fit` no treino, `transform` no teste

O vetorizador também **aprende**: ele monta o vocabulário a partir dos textos que recebe. Se ele visse o teste, o vocabulário "espiaria" a prova. Por isso há dois métodos diferentes:

| Método | Onde usar | O que faz |
|---|---|---|
| `fit_transform` | **só no treino** | aprende o vocabulário **e** transforma os textos em números |
| `transform` | no teste (e em textos novos) | só transforma, usando o vocabulário **já aprendido** |

Palavras do teste que não existem no vocabulário do treino são simplesmente ignoradas.

Vamos guardar só as **1000 palavras mais frequentes** (`max_features=1000`). Assim a tabela fica menor, a árvore treina mais rápido e as regras dela ficam fáceis de ler.

🧭 **Complete** com o método certo em cada linha.

In [ ]:
vetorizador = CountVectorizer(max_features=1000)

X_treino_vet = vetorizador.___(X_treino)
X_teste_vet  = vetorizador.___(X_teste)

palavras = vetorizador.get_feature_names_out()
print("Tamanho do vocabulário:", len(palavras), "palavras")
print("Tabela de treino:", X_treino_vet.shape, "(textos × palavras)")
print("Tabela de teste: ", X_teste_vet.shape)
print("Algumas palavras do vocabulário:", list(palavras[300:310]))

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
X_treino_vet = vetorizador.fit_transform(X_treino)
X_teste_vet  = vetorizador.transform(X_teste)
```
</details>

✍️ **Responda:**
1. As duas tabelas têm o **mesmo número de colunas**. Por que isso é obrigatório para o modelo funcionar?
2. O que aconteceria se usássemos `fit_transform` também no teste?

**✍️ Respostas da equipe**

1.
2.

## Etapa 6 · Treinar uma árvore de decisão

Uma **árvore de decisão** aprende uma sequência de **perguntas de sim ou não**, como num jogo de adivinhação. Com texto, cada pergunta é sobre uma palavra, por exemplo *"a palavra `defeito` aparece no texto?"*. A árvore escolhe sozinha **quais palavras perguntar e em que ordem**, buscando as perguntas que melhor separam `sim` de `não`.

Vamos começar com uma árvore **pequena**, de profundidade 3 (no máximo 3 perguntas seguidas), para conseguir ler o desenho dela.

Treinar, no scikit-learn, é sempre o mesmo gesto: `modelo.fit(X, y)`.

🧭 **Complete** com a tabela de **treino** já vetorizada e com os **rótulos de treino**.

In [ ]:
arvore = DecisionTreeClassifier(max_depth=3, random_state=SEMENTE)
arvore.fit(___, ___)

print("Árvore treinada! Classes que ela conhece:", arvore.classes_)

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
arvore.fit(X_treino_vet, y_treino)
```
</details>

### Lendo as regras da árvore
▶️ **Rode** para desenhar a árvore. Como ler cada caixa:
- a **primeira linha** é a pergunta. `não <= 0.5` quer dizer *"a palavra `não` aparece **zero** vezes?"*. Se a resposta é **verdadeira**, siga para a **esquerda**; se é falsa (a palavra aparece), siga para a **direita**;
- `samples` é quantas avaliações de treino chegaram até ali;
- `value` é quantas delas são `[não, sim]`;
- `class` é o que a árvore responde se o texto parar naquela caixa. A cor mostra a classe: quanto mais forte, mais "certa" está a árvore.

In [ ]:
plt.figure(figsize=(18, 7))
plot_tree(arvore, feature_names=palavras, class_names=arvore.classes_,
          filled=True, rounded=True, fontsize=10, impurity=False)
plt.show()

✍️ **Responda** olhando o desenho:
1. Qual é a **primeira pergunta** da árvore? Por que vocês acham que ela escolheu essa palavra?
2. Sigam o caminho da frase *"não gostei, veio com defeito"*. Em que caixa ela termina e qual a resposta?
3. Aparecem as palavras `não` **e** `nao` (sem acento). Para a árvore, elas são palavras diferentes! Que etapa de preparação do texto resolveria isso? *(Dica: o `CountVectorizer` aceita `strip_accents="unicode"`, que tira os acentos de tudo.)*

**✍️ Respostas da equipe**

1.
2.
3.

### Testando com frases nossas
Um texto novo passa pelo **mesmo** caminho: `transform` no vetorizador e depois `predict` no modelo.

🧭 **Complete:** escrevam **três frases suas** na lista, no lugar dos `"___"`: uma claramente positiva, uma claramente negativa e uma **irônica** (por exemplo, *"Maravilha, chegou quebrado de novo"*). Depois rodem.

In [ ]:
minhas_frases = [
    "Produto excelente, chegou antes do prazo",
    "Não gostei, veio com defeito",
    "___",
    "___",
    "___",
]

previsto = arvore.predict(vetorizador.transform(minhas_frases))
for frase, p in zip(minhas_frases, previsto):
    print(f"{p:>4}  ←  {frase}")

✍️ **Responda:** a árvore acertou a frase irônica? Usando o desenho da árvore, expliquem **por que** ela respondeu o que respondeu.

**✍️ Resposta da equipe:**

## Etapa 7 · Avaliar o modelo no teste

Chegou a hora da prova. Pedimos à árvore uma previsão para cada texto de teste e comparamos com a resposta verdadeira (`y_teste`).

🧭 **Complete** com o método que **faz previsões** (o par do `fit`).

In [ ]:
previsoes = arvore.___(X_teste_vet)

acuracia = accuracy_score(y_teste, previsoes)
linha_de_base = (y_teste == "sim").mean()   # o "modelo preguiçoso" que sempre diz sim

print(f"Acurácia da árvore (profundidade 3): {acuracia:.3f}")
print(f"Linha de base (sempre 'sim'):        {linha_de_base:.3f}")

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
previsoes = arvore.predict(X_teste_vet)
```
</details>

### A matriz de confusão
A acurácia sozinha não diz **que tipo** de erro o modelo comete. A matriz de confusão mostra:
- **linhas** = o que o cliente respondeu de verdade;
- **colunas** = o que o modelo previu.

Para o SAC, a classe que interessa é **`não`** (o cliente insatisfeito).

▶️ **Rode.**

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_teste, previsoes, labels=["não", "sim"], cmap="Blues")
plt.title("Árvore de decisão (profundidade 3)")
plt.xlabel("Previsto pelo modelo")
plt.ylabel("Resposta real do cliente")
plt.show()

### Precisão e revocação da classe `não`
Duas perguntas que o SAC faria, ditas em português simples:

- **Revocação** (*recall*): dos clientes que estavam **realmente insatisfeitos**, quantos o modelo **encontrou**? Revocação baixa = cliente insatisfeito esquecido na fila.
- **Precisão** (*precision*): dos textos que o modelo **marcou** como `não`, quantos **eram mesmo**? Precisão baixa = SAC perdendo tempo com cliente satisfeito.

▶️ **Rode** o relatório completo. Procurem a linha `não`.

In [ ]:
print(classification_report(y_teste, previsoes, digits=3))

✍️ **Responda** com os números da sua tela:

| Métrica | Valor |
|---|---|
| Acurácia | |
| Linha de base | |
| Precisão de `não` | |
| Revocação de `não` | |

1. A árvore é melhor que a linha de base? Por quanto?
2. Para o SAC, qual erro custa mais caro: marcar como `não` um cliente satisfeito (**falso positivo**) ou deixar passar um cliente insatisfeito (**falso negativo**)? Então, qual das duas métricas o SAC deve olhar primeiro?

**✍️ Respostas da equipe**

1.
2.

## Etapa 8 · Qual a profundidade certa? Subajuste e sobreajuste

Uma árvore de profundidade 3 faz só 3 perguntas: talvez seja **simples demais** (*subajuste*). Uma árvore sem limite faz perguntas até "decorar" cada avaliação do treino: ela vai ficar **perfeita no treino**, mas será que acerta textos novos? Esse é o **sobreajuste** da Aula 07.

Vamos treinar árvores de várias profundidades e medir a acurácia **no treino e no teste**. `None` quer dizer "sem limite".

🧭 **Complete** dentro do laço: a profundidade da vez (a variável `p`) e os dados de treino do `fit`.

In [ ]:
profundidades = [1, 2, 3, 5, 10, 20, None]
tabela = []

for p in profundidades:
    modelo = DecisionTreeClassifier(max_depth=___, random_state=SEMENTE)
    modelo.fit(___, ___)
    tabela.append({
        "profundidade": str(p),
        "acurácia no treino": accuracy_score(y_treino, modelo.predict(X_treino_vet)),
        "acurácia no teste":  accuracy_score(y_teste, modelo.predict(X_teste_vet)),
    })

tabela = pd.DataFrame(tabela).round(3)
tabela

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
    modelo = DecisionTreeClassifier(max_depth=p, random_state=SEMENTE)
    modelo.fit(X_treino_vet, y_treino)
```
</details>

▶️ **Rode** para ver a tabela como gráfico.

In [ ]:
ax = tabela.plot(x="profundidade", y=["acurácia no treino", "acurácia no teste"], marker="o", figsize=(8, 4))
ax.axhline(linha_de_base, color="gray", linestyle=":", label="linha de base")
ax.set_ylabel("acurácia")
ax.set_title("Treino × teste conforme a árvore cresce")
ax.legend()
plt.show()

✍️ **Responda:**
1. O que acontece com a acurácia **no treino** quando a árvore cresce sem limite? E **no teste**?
2. A partir de que profundidade a árvore começa a **decorar** o treino em vez de aprender? Como vocês perceberam isso no gráfico?
3. Qual profundidade vocês escolheriam? Lembrem: a escolha é pelo **teste**, nunca pelo treino.

**✍️ Respostas da equipe**

1.
2.
3.

### A árvore escolhida
🧭 **Complete:** coloquem em `PROFUNDIDADE` o valor que a equipe escolheu na pergunta 3 e rodem. Comparem a matriz de confusão e a revocação de `não` com as da Etapa 7.

In [ ]:
PROFUNDIDADE = ___

arvore_final = DecisionTreeClassifier(max_depth=PROFUNDIDADE, random_state=SEMENTE)
arvore_final.fit(X_treino_vet, y_treino)
previsoes_final = arvore_final.predict(X_teste_vet)

ConfusionMatrixDisplay.from_predictions(y_teste, previsoes_final, labels=["não", "sim"], cmap="Blues")
plt.title(f"Árvore de decisão (profundidade {PROFUNDIDADE})")
plt.xlabel("Previsto pelo modelo")
plt.ylabel("Resposta real do cliente")
plt.show()
print(classification_report(y_teste, previsoes_final, digits=3))

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
PROFUNDIDADE = 10
# No teste, 10 e 20 quase empatam (≈ 0,85), mas a árvore de 10 é bem menor.
# Com empate prático, preferimos o modelo mais simples. Sem limite (None), o treino
# chega a 1,00 e o teste CAI: é o sobreajuste.
```
</details>

### Quais palavras a árvore mais usou?
A árvore final é grande demais para desenhar, mas podemos perguntar a ela **quais palavras mais ajudaram** a separar `sim` de `não` (a *importância* de cada palavra).

▶️ **Rode.**

In [ ]:
importancia = pd.Series(arvore_final.feature_importances_, index=palavras)
importancia.nlargest(15).sort_values().plot(kind="barh", figsize=(7, 5), color="tab:blue",
                                            title="As 15 palavras mais importantes para a árvore")
plt.xlabel("importância")
plt.show()

✍️ **Responda:** as palavras fazem sentido para decidir se o cliente recomenda o produto? Alguma surpreendeu a equipe?

**✍️ Resposta da equipe:**

## Etapa 9 · Onde o modelo erra?
Uma métrica alta não basta: precisamos saber **onde** o modelo erra.

▶️ **Rode** para ver algumas avaliações do teste em que a árvore final **errou**.

In [ ]:
teste = pd.DataFrame({"texto": X_teste, "real": y_teste, "previsto": previsoes_final})
erros = teste[teste["real"] != teste["previsto"]]

print(f"A árvore errou {len(erros)} de {len(teste)} avaliações de teste.")
erros.sample(8, random_state=SEMENTE)

✍️ **Responda:** leiam os erros e classifiquem pelo menos **quatro** deles em uma destas causas:

| Causa | Exemplo típico |
|---|---|
| **Ironia** | "Ótimo, quebrou no segundo dia" |
| **Negação** | "Não é ruim, gostei" |
| **Gíria ou erro de digitação** | "produto top dmais", "nao recomemdo" |
| **Cliente contraditório** | elogia o texto inteiro e responde que não recomenda |
| **Outro** | descrevam |

**✍️ Respostas da equipe**

| Trecho do texto | Real | Previsto | Causa |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |

> **Os limites do saco de palavras.** Ele não enxerga ordem, ironia nem contexto: só conta palavras. A resposta da área para isso são as representações que capturam o **sentido** das palavras e frases, os ***embeddings***, e os **grandes modelos de linguagem** (LLMs), que vimos na Aula 04. O caminho de hoje (dados → treino e teste → vetorização → modelo → avaliação) continua o mesmo: só muda a forma de transformar texto em números.

## Etapa 10 · Do que os clientes reclamam? Clusterização com k-means

Agora a **Pergunta 2**. O SAC quer saber os **assuntos** das reclamações, mas ninguém rotulou as avaliações por assunto. Não há `y`: é um problema **não supervisionado**.

A **clusterização** junta em um mesmo grupo (*cluster*) os textos **parecidos entre si**. O **k-means** funciona assim:
1. nós escolhemos **quantos grupos** queremos (o `k`);
2. ele sorteia `k` pontos de partida e coloca cada texto no grupo do ponto mais próximo;
3. recalcula o "centro" de cada grupo e repete até os grupos pararem de mudar.

No fim, **quem dá nome aos grupos somos nós**, lendo as palavras mais típicas de cada um.

▶️ **Rode** para separar só as avaliações de clientes que **não** recomendariam o produto.

In [ ]:
negativas = df.loc[df["recomenda"] == "não", "texto"].reset_index(drop=True)
print(len(negativas), "avaliações negativas")

### Dois cuidados antes de agrupar
**1. Stopwords.** Palavras como "de", "o", "que" aparecem em **todos** os textos e não dizem nada sobre o assunto. Elas se chamam ***stopwords*** e vamos tirá-las com uma lista feita à mão. Repare que, aqui, `não` **entra** na lista: todas estas avaliações são negativas, então `não` não ajuda a separar um assunto do outro. Na árvore, ao contrário, `não` era a palavra mais importante!

**2. TF-IDF.** Em vez de só contar as palavras, damos **peso menor** às palavras que aparecem em muitos textos e **peso maior** às mais específicas, como "atraso" ou "quebrado". É o `TfidfVectorizer`, que funciona igual ao `CountVectorizer`. Sem esse ajuste, o k-means costuma jogar quase tudo num grupo só.

▶️ **Rode.**

In [ ]:
STOPWORDS = """a o as os e é de do da dos das em no na nos nas um uma uns umas para pra por pelo pela com
que se ao aos à às eu me meu minha meus minhas ele ela eles isso esse essa este esta isto
foi ser são está estou estava era tem ter tinha mas mais muito muita como já também só
ou nem quando até vai vou porque pois lo la seu sua seus suas tá fiz fica ficou ainda
não nao produto produtos comprei compra americanas""".split()

vet_temas = TfidfVectorizer(stop_words=STOPWORDS, min_df=5)   # min_df=5: ignora palavras que aparecem em menos de 5 textos
X_temas = vet_temas.fit_transform(negativas)
termos = vet_temas.get_feature_names_out()

print("Tabela para o k-means:", X_temas.shape, "(avaliações × palavras)")

### Rodando o k-means
🧭 **Complete:** escolham `K = 5` grupos para começar, e passem a tabela `X_temas` para o `fit`. Não há `y` no `fit`: o k-means **não vê rótulo nenhum**.

In [ ]:
K = ___

kmeans = KMeans(n_clusters=K, random_state=SEMENTE, n_init=10)
kmeans.fit(___)

grupos = pd.Series(kmeans.labels_, name="grupo")
print("Avaliações em cada grupo:")
print(grupos.value_counts().sort_index())

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
K = 5
kmeans.fit(X_temas)
```
</details>

▶️ **Rode** para ver as **10 palavras mais típicas** de cada grupo (as de maior peso no centro do grupo) e dois exemplos de cada.

In [ ]:
for g, centro in enumerate(kmeans.cluster_centers_):
    top = termos[centro.argsort()[::-1][:10]]
    print(f"Grupo {g} ({(grupos == g).sum()} avaliações): {', '.join(top)}")
    for texto in negativas[grupos == g].sample(2, random_state=SEMENTE):
        print("    •", texto[:150])
    print()

✍️ **Responda:**
1. Deem um **nome** a cada grupo (por exemplo, "entrega atrasada"). Preencham a tabela abaixo.
2. Algum grupo ficou "misturado", sem um assunto claro? Por que isso pode acontecer?
3. Mudem `K` para **3** e depois para **8**, rodando de novo as duas células do k-means. Os assuntos ficam mais claros ou mais confusos? Qual `K` vocês entregariam ao SAC?

| Grupo | Palavras principais | Nome dado pela equipe |
|---|---|---|
| 0 | | |
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |

**✍️ Respostas da equipe**

2.
3.

> **Árvore × k-means.** A árvore **aprendeu com as respostas** dos clientes (`sim`/`não`) e pode ser avaliada com acurácia, precisão e revocação. O k-means **não tinha respostas**: ele só juntou o que é parecido, e não existe uma "acurácia" para conferir. Quem julga se os grupos são úteis é o analista, junto com o SAC.

## Etapa 11 · Registro do experimento no CRISP-DM

Fechem o ciclo como na Unidade II: **✍️ completem** a coluna da direita com as decisões e os números da equipe.

| Fase do CRISP-DM | Pergunta-guia | Registro da equipe |
|---|---|---|
| 1. Entendimento do negócio | Quais são as duas perguntas do SAC e qual erro custa mais? | |
| 2. Entendimento dos dados | De onde vêm os dados, quantas avaliações, qual a proporção de `não`? | |
| 3. Preparação dos dados | O que foi removido, traduzido ou sorteado? Quais stopwords? | |
| 4. Modelagem | Que profundidade de árvore e que `K` foram escolhidos, e por quê? | |
| 5. Avaliação | Quanto a árvore superou a linha de base? Os grupos do k-means fazem sentido? | |
| 6. Implantação | Como o SAC usaria a árvore e os grupos no dia a dia? Que cuidado manter? | |

## 🏁 Desafio extra (para quem terminar antes)

Escolham **um** dos itens e façam em células novas abaixo.

1. **Tirar os acentos:** refaçam as Etapas 5 a 7 com `CountVectorizer(max_features=1000, strip_accents="unicode")`. As palavras `não` e `nao` viram uma só? A acurácia e a revocação de `não` mudam?
2. **Dar mais peso aos insatisfeitos:** treinem a árvore final com `class_weight="balanced"`. Como há muito mais `sim` que `não`, essa opção faz a árvore prestar mais atenção na classe rara. O que acontece com a **revocação** e com a **precisão** de `não`?
3. **Só o título:** troquem `df["texto"]` por `df["titulo"].fillna("")` na Etapa 4. O título curto prevê tão bem quanto o texto completo?

## Encerramento

**O que fizemos hoje:**
1. Traduzimos as perguntas do negócio em problemas de aprendizado: um **supervisionado** (recomenda?) e um **não supervisionado** (do que reclamam?).
2. Transformamos texto em números com o **saco de palavras**, ajustando o vetorizador **só no treino**.
3. Treinamos uma **árvore de decisão**, lemos as regras dela e a avaliamos contra uma **linha de base**, escolhendo a métrica pelo **custo do erro**.
4. Vimos o **sobreajuste** na prática: a árvore sem limite decora o treino e piora no teste.
5. Agrupamos as reclamações com **k-means** e demos nome aos grupos.
6. Lemos os erros e encontramos os limites do saco de palavras: **ironia, negação e gírias**.

**Próxima aula (08):** a revocação que vocês mediram em 5 mil avaliações de teste é uma **estimativa amostral**. Com outro sorteio, o número mudaria um pouco. Vamos aprender a colocar uma **margem de erro** nesse número.